<a href="https://colab.research.google.com/github/Rayoyo/NLP-Translator-JA-EN/blob/main/Project_NLP_Translator_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Enviroment setup

In [ ]:
# Verifica GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q sentencepiece sacrebleu transformers gradio datasets tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import sys
import os

---
## 2. Clone repository from Github

In [ ]:
# Clona la repo (oppure carica i file manualmente)
# !git clone https://github.com/TUO_USERNAME/PROGETTO-TRADUTTORE.git
# sys.path.append('/content/PROGETTO-TRADUTTORE')

# OPPURE carica src/ manualmente su Colab
# Assumiamo che i file siano in /content/drive/MyDrive/progetto/

In [ ]:
!git clone https://github.com/Rayoyo/NLP-Translator-JA-EN.git
%cd NLP-Translator-JA-EN

import sys
sys.path.append('/content/NLP-Translator-JA-EN')

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/university/Project-NLP_Translator"
DATA_PATH = f"{PROJECT_PATH}/data/processed"
MODELS_PATH = f"{PROJECT_PATH}/models"

os.makedirs(MODELS_PATH, exist_ok=True)

EN_FILE = f"{DATA_PATH}/english.txt"
JP_FILE = f"{DATA_PATH}/japanese.txt"

print(f"EN: {os.path.exists(EN_FILE)} ({os.path.getsize(EN_FILE)/1e9:.2f} GB)")
print(f"JP: {os.path.exists(JP_FILE)} ({os.path.getsize(JP_FILE)/1e9:.2f} GB)")

---
## 3. Import custom models

In [ ]:
Assumendo che src/ sia in PYTHONPATH
# Se usi GitHub: sys.path.append('/content/PROGETTO-TRADUTTORE')

In [ ]:
from src.tokenizer import setup_tokenizers
from src.dataset import create_dataloaders
from src.transformer import Transformer, count_parameters
from src.train import Trainer, get_scheduler
from src.evaluate import extract_test_set, evaluate_models

# Parameters
VOCAB_SIZE = 32000
BATCH_SIZE = 16  # can try 32
D_MODEL = 512
N_HEADS = 8
N_LAYERS = 6
D_FF = 2048
MAX_SAMPLES = None  # None = all dataset, or 1000000 for initial test

DEVICE = 'cuda'

---
## 4. Tokenizer

In [ ]:
# only one time, then comment it
sp_en, sp_jp = setup_tokenizers(
    EN_FILE,
    JP_FILE,
    vocab_size=VOCAB_SIZE,
    model_dir=MODELS_PATH
)

---
## 5. Dataset e DataLoader (LAZY)

In [ ]:
# Estrai test set PRIMA di creare il dataloader (escludendo quegli indici)
# Oppure semplicemente usa file separati per train/test

In [ ]:
train_loader = create_dataloaders(
    EN_FILE,
    JP_FILE,
    sp_en,
    sp_jp,
    batch_size=BATCH_SIZE,
    num_workers=2,
    max_samples=MAX_SAMPLES  # For sanity check: at the start use 100000
)

print(f"Train batches: {len(train_loader)}")

---
## 6. Model

In [ ]:
model = Transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_encoder_layers=N_LAYERS,
    n_decoder_layers=N_LAYERS,
    d_ff=D_FF,
    dropout=0.1,
    pad_idx=0
).to(DEVICE)

print(f"Model parameters: {count_parameters(model):,}")

In [ ]:
# Optimizer & scheduler
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)
scheduler = get_scheduler(optimizer, D_MODEL, warmup_steps=4000)

---
## 7. Training

In [ ]:
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    save_dir=MODELS_PATH,
    log_interval=100
)

# For sanity check: 2-3 epoch on 10% data
# For completed training: 10-20 epochs
trainer.fit(n_epochs=10)

---
## 8. Translation test

In [ ]:
def translate(text, direction="en-jp"):
    model.eval()
    with torch.no_grad():
        sp_src, sp_tgt = (sp_en, sp_jp) if direction == "en-jp" else (sp_jp, sp_en)
        src_ids = sp_src.encode(text, out_type=int, add_bos=True, add_eos=True)
        src_tensor = torch.tensor([src_ids], dtype=torch.long).to(DEVICE)
        out = model.translate(src_tensor, max_len=50, bos_id=2, eos_id=3)
        out_ids = [id for id in out[0].cpu().tolist() if id not in [0, 2, 3]]
        return sp_tgt.decode(out_ids)

print(translate("Hello, how are you?"))

---
## 9. Extract Test set & Comparison

In [ ]:
# Extract 1000 phrases for test model
test_en, test_jp, test_indices = extract_test_set(EN_FILE, JP_FILE, n=1000)

# Load best model
trainer.load_checkpoint(f"{MODELS_PATH}/best_model.pt")

# Evaluation EN -> JP
results_en_jp = evaluate_models(
    trainer.model,
    sp_en, sp_jp,
    test_en, test_jp,
    direction="en-jp"
)

# Evaluation JP -> EN
results_jp_en = evaluate_models(
    trainer.model,
    sp_en, sp_jp,
    test_en, test_jp,
    direction="jp-en"
)

# Save results
import json
with open(f"{MODELS_PATH}/evaluation_results.json", 'w') as f:
    json.dump({
        'en_jp': {
            'my_bleu': results_en_jp['my_bleu'],
            'pretrained_bleu': results_en_jp['pretrained_bleu']
        },
        'jp_en': {
            'my_bleu': results_jp_en['my_bleu'],
            'pretrained_bleu': results_jp_en['pretrained_bleu']
        }
    }, f, indent=2)

---
## 10. GUI on Colab

In [ ]:
# from src.gui import TranslatorApp

# app = TranslatorApp(trainer.model, sp_en, sp_jp, device='cuda')
# app.launch(share=True)  # Create temporary public link